# Test 2025's spend and revenue

*Data Analysis in Python · Topic 8.3*

The CMO's slide for the budget meeting says "ad spend drives revenue", and the evidence is a correlation for 2025. Draw it, calculate it, and then test it: how much of it is left in two months without a sale or a holiday?

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that builds the `daily` table: one row per day, with the dates as the index (topic 6.1.'s steps), and a `spend` column, the three ad platforms' spend that day (lesson 5.'s steps).
2. In the first cell marked ✏️, create `scatter_ax`: the Axes of a scatter plot of the `daily` table, with `spend` on the x axis and `revenue` on the y axis, and a label on both axes.
3. In the second cell marked ✏️, create:
   - the `correlations` table: the correlation of every pair of the `daily` table's columns
   - `heatmap_ax`: the Axes of a heatmap of the `correlations` table, with the values written in the cells
   - `spend_revenue_r`: the correlation of the `daily` table's `spend` and `revenue`
4. In the last cell marked ✏️, create `quiet_r`: the correlation of `spend` and `revenue` in August and September 2025 only, two months without a sale or a holiday.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `scatter_ax` has 365 dots, spend on the x axis and revenue on the y axis, with both axes labelled. The `correlations` table is 4 × 4, and `heatmap_ax` shows its 16 values. `spend_revenue_r` is about `0.7777`. `quiet_r` is about `-0.0119`.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"2025-{month:02d}" for month in range(1, 13)]
START, END = "2025-01-01", "2025-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# One row per day (topic 6.1), with the three platforms' ad spend that day (lesson 5). Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()

google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
spend = pd.concat([
    pd.DataFrame({"date": google["date"], "spend": google["cost_micros"] / 1_000_000}),
    pd.DataFrame({"date": meta["Reporting starts"], "spend": meta["Amount spent (USD)"]}),
    pd.DataFrame({"date": youtube["Day"], "spend": youtube["Cost"]}),
])
daily["spend"] = spend.groupby("date")["spend"].sum()
daily.shape

## The scatter plot

In [ ]:
# ✏️ Create scatter_ax


## The correlations

In [ ]:
# ✏️ Create correlations, heatmap_ax and spend_revenue_r


## Two quiet months

In [ ]:
# ✏️ Create quiet_r


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib.axes
    import numpy as np
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    SPEND, REVENUE = 318705.08, 3005212.51
    scatter_ax = found.get("scatter_ax")
    if not isinstance(scatter_ax, matplotlib.axes.Axes):
        results.append((False, f"scatter_ax should be the Axes of your scatter plot: fig, scatter_ax = plt.subplots(), then scatter_ax.scatter(daily[\"spend\"], daily[\"revenue\"]). It is a {type(scatter_ax).__name__}."))
    else:
        dots = [collection.get_offsets() for collection in scatter_ax.collections if len(collection.get_offsets())]
        points = np.concatenate(dots) if dots else np.empty((0, 2))
        if len(points) == 0:
            results.append((False, "scatter_ax has no dots on it. Draw them: scatter_ax.scatter(daily[\"spend\"], daily[\"revenue\"])."))
        elif len(points) != 365:
            results.append((False, f"scatter_ax should have 365 dots, one per day of the daily table. It has {len(points):,}."))
        elif abs(points[:, 0].sum() - REVENUE) < 1 and abs(points[:, 1].sum() - SPEND) < 1:
            results.append((False, "scatter_ax has revenue on the x axis and spend on the y axis. Put the suspected cause, spend, on the x axis: scatter_ax.scatter(daily[\"spend\"], daily[\"revenue\"])."))
        elif abs(points[:, 0].sum() - SPEND) > 1 or abs(points[:, 1].sum() - REVENUE) > 1:
            results.append((False, "scatter_ax's dots should be each day's spend (x) and revenue (y), from the daily table."))
        elif not scatter_ax.get_xlabel().strip() or not scatter_ax.get_ylabel().strip():
            results.append((False, "scatter_ax needs a label on both axes: scatter_ax.set_xlabel(\"Ad spend ($)\") and scatter_ax.set_ylabel(\"Revenue ($)\")."))
        else:
            results.append((True, "The scatter plot has 365 days, spend on the x axis and revenue on the y axis, both labelled."))

    correlations = found.get("correlations")
    heatmap_ax = found.get("heatmap_ax")
    r = found.get("spend_revenue_r")
    if not isinstance(correlations, pd.DataFrame):
        results.append((False, "correlations should be a DataFrame: daily.corr()."))
    elif "spend" not in correlations.columns:
        results.append((False, "The correlations table has no spend column: it comes from the sessions table. Correlate the daily table: daily.corr()."))
    elif correlations.shape != (4, 4) or abs(correlations.loc["spend", "revenue"] - 0.7777) > 0.0001:
        results.append((False, f"The correlations table should be 4 × 4, every pair of the daily table's columns, with spend and revenue at about 0.7777. It is {correlations.shape[0]} × {correlations.shape[1]}."))
    elif not isinstance(heatmap_ax, matplotlib.axes.Axes):
        results.append((False, f"heatmap_ax should be the Axes of your heatmap: fig, heatmap_ax = plt.subplots(), then sns.heatmap(correlations, annot=True, ax=heatmap_ax). It is a {type(heatmap_ax).__name__}."))
    else:
        cells = [np.ma.compressed(item.get_array()) for item in list(heatmap_ax.collections) + list(heatmap_ax.images) if item.get_array() is not None]
        shown = sorted(np.concatenate(cells)) if cells else []
        expected = sorted(correlations.to_numpy().ravel())
        if len(shown) != 16 or not np.allclose(shown, expected):
            results.append((False, f"heatmap_ax should show the correlations table's 16 values. It shows {len(shown)}. Draw it with sns.heatmap(correlations, annot=True, ax=heatmap_ax)."))
        elif not heatmap_ax.texts:
            results.append((False, "heatmap_ax's cells have no numbers in them. Add annot=True, and fmt=\".2f\" for two decimals."))
        elif not number(r):
            results.append((False, f"spend_revenue_r should be a number: daily[\"spend\"].corr(daily[\"revenue\"]). It is {r!r}."))
        elif abs(r - 0.9063) < 0.0001:
            results.append((False, "spend_revenue_r is 0.9063: that's spend and sessions. Correlate spend with revenue."))
        elif abs(r - 0.7834) < 0.0001:
            results.append((False, "spend_revenue_r is 0.7834: that's spend and transactions. Correlate spend with revenue."))
        elif abs(r - 0.7777) > 0.0001:
            results.append((False, f"spend_revenue_r should be the correlation of the daily table's spend and revenue, about 0.7777. It is {r:.4f}."))
        else:
            results.append((True, f"Spend and revenue correlate at {r:.4f} over the year, and the heatmap shows all 16 correlations, transactions and revenue the strongest pair at {correlations.loc['transactions', 'revenue']:.4f}."))

    quiet = found.get("quiet_r")
    if not number(quiet):
        results.append((False, f"quiet_r should be a number: the correlation of spend and revenue in the daily table's rows from August 1 to September 30, 2025. It is {quiet!r}."))
    elif abs(quiet - 0.7777) < 0.0001:
        results.append((False, "quiet_r is 0.7777, the whole year's. Select August and September first: daily.loc[\"2025-08-01\":\"2025-09-30\"]."))
    elif abs(quiet - 0.0013) < 0.0001 or abs(quiet - 0.0456) < 0.0001:
        results.append((False, "quiet_r is one month's correlation. Use both months: daily.loc[\"2025-08-01\":\"2025-09-30\"]."))
    elif abs(quiet + 0.0119) > 0.0001:
        results.append((False, f"quiet_r should be the correlation of spend and revenue from August 1 to September 30, 2025, about -0.0119. It is {quiet:.4f}."))
    else:
        results.append((True, f"In August and September, spend and revenue correlate at {quiet:.4f}, against {0.7777:.4f} for the year: most of the year's correlation is the season, which moves both."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
fig, scatter_ax = plt.subplots(figsize=(6, 5))
scatter_ax.scatter(daily["spend"], daily["revenue"], alpha=0.5)
scatter_ax.set_title("Daily ad spend and revenue, 2025")
scatter_ax.set_xlabel("Ad spend ($)")
scatter_ax.set_ylabel("Revenue ($)")
plt.show()
```

```python
correlations = daily.corr()
fig, heatmap_ax = plt.subplots(figsize=(5, 4))
sns.heatmap(correlations, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu", ax=heatmap_ax)
heatmap_ax.set_title("Correlations, daily, 2025")
plt.show()
spend_revenue_r = daily["spend"].corr(daily["revenue"])
spend_revenue_r
```

```python
quiet = daily.loc["2025-08-01":"2025-09-30"]
quiet_r = quiet["spend"].corr(quiet["revenue"])
quiet_r
```

</details>